# Predictive Analytics for Student Performance
## Phase 4: Model Evaluation & Interpretation

### Objective
Evaluate the model across standard regression metrics (MAE, MSE, RMSE, R²), plot residuals, and analyze feature importance.


In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load artifacts
with open('../models/random_forest_model.pkl', 'rb') as f:
    rf = pickle.load(f)
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('../models/label_encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

# Load data and evaluate
df = pd.read_csv('../data/student_performance.csv')
for col in ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']:
    df[col] = df[col].fillna(df[col].mode()[0])

categorical_cols = list(encoders.keys())
numerical_cols = ['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

for col in categorical_cols:
    df[col] = encoders[col].transform(df[col].astype(str))

features = numerical_cols + categorical_cols
X = df[features]
y = df['Exam_Score']

X_scaled = scaler.transform(X)
preds = rf.predict(X_scaled)


### 1. Overall Performance Metrics


In [ ]:
mae = mean_absolute_error(y, preds)
mse = mean_squared_error(y, preds)
rmse = np.sqrt(mse)
r2 = r2_score(y, preds)

print("Evaluation Summary:")
print(f"  R2 Score: {r2:.4f}")
print(f"  MAE:      {mae:.4f}")
print(f"  RMSE:     {rmse:.4f}")


### 2. Actual vs. Predicted Exam Scores


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y[:200], preds[:200], alpha=0.6, color='#6C63FF')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Actual Exam Score')
plt.ylabel('Predicted Exam Score')
plt.title('Actual vs. Predicted Exam Scores')
plt.show()


### 3. Feature Importance Analysis


In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.plot(kind='bar', color='#4CAF50')
plt.title('Feature Importances in Predicting Exam Score')
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
